# 06 — Deterministic Graph Layer (Milestone M3)

**DSML stage:** modeling. Loads everything that needs **no LLM** — pure API-derived facts:

- `Company` nodes (the 14-company universe, CIKs from notebook 01)
- `Filing` nodes + `FILED` edges (from the manifest)
- `FilingSection` nodes + `HAS_SECTION` edges (from notebook 03's segmentation)
- `Metric` nodes + `REPORTS_METRIC` edges (from notebook 02's curated XBRL parquet)

All loaders are idempotent `MERGE`s — re-running never duplicates.

In [1]:
import json
import os
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv
from neo4j import GraphDatabase

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
load_dotenv(PROJECT_ROOT / ".env")

driver = GraphDatabase.driver(
    os.environ["NEO4J_URI"], auth=(os.environ["NEO4J_USER"], os.environ["NEO4J_PASSWORD"])
)
driver.verify_connectivity()  # run notebook 05 first if this fails

MANIFEST = json.loads((PROJECT_ROOT / "data/raw/edgar/NVDA/manifest.json").read_text())
raw_tickers = json.loads((PROJECT_ROOT / "data/raw/edgar/company_tickers.json").read_text())
ticker_to_cik = {row["ticker"]: row["cik_str"] for row in raw_tickers.values()}

UNIVERSE = {  # ticker: (canonical name, tier) — Samsung has no CIK (not an SEC filer)
    "MSFT": ("Microsoft", "Hyperscaler"), "AMZN": ("Amazon", "Hyperscaler"),
    "GOOGL": ("Alphabet", "Hyperscaler"), "META": ("Meta", "Hyperscaler"),
    "NVDA": ("Nvidia", "Silicon Designer"), "AMD": ("AMD", "Silicon Designer"),
    "AVGO": ("Broadcom", "Silicon Designer"), "QCOM": ("Qualcomm", "Silicon Designer"),
    "INTC": ("Intel", "IDM"), "TSM": ("TSMC", "Manufacturer"), "ASML": ("ASML", "Manufacturer"),
    "MU": ("Micron", "Memory"), "SSNLF": ("Samsung", "Memory"), "AAPL": ("Apple", "Ecosystem Anchor"),
}

## 1. Company nodes

Companies without a CIK (Samsung) get a synthetic negative key so the unique constraint still holds.

In [2]:
company_rows = []
synthetic = -1
for ticker, (name, tier) in UNIVERSE.items():
    cik = ticker_to_cik.get(ticker)
    if cik is None:
        cik, synthetic = synthetic, synthetic - 1  # non-SEC-filer
    company_rows.append({"cik": cik, "ticker": ticker, "name": name, "tier": tier,
                          "sec_filer": ticker in ticker_to_cik})

with driver.session() as session:
    session.run(
        """UNWIND $rows AS row
        MERGE (c:Company {cik: row.cik})
        SET c.ticker = row.ticker, c.name = row.name, c.tier = row.tier, c.sec_filer = row.sec_filer""",
        rows=company_rows,
    )
    count = session.run("MATCH (c:Company) RETURN count(c) AS n").single()["n"]
print(f"{count} Company nodes")

14 Company nodes


## 2. Filing nodes + FILED edges, FilingSection nodes + HAS_SECTION edges

In [3]:
section_texts = pd.read_parquet(PROJECT_ROOT / "data/interim/section_texts/nvda_section_texts.parquet")
section_rows = [
    {
        "section_key": f"{r.accession_no}:{r.section_id}",
        "accession_no": r.accession_no,
        "section_id": r.section_id,
        "title": r.section_title,
        "n_chars": int(r.n_chars),
    }
    for r in section_texts.itertuples()
]

with driver.session() as session:
    session.run(
        """UNWIND $rows AS row
        MATCH (c:Company {ticker: row.ticker})
        MERGE (f:Filing {accession_no: row.accession_no})
        SET f.form = row.form, f.filing_date = date(row.filing_date), f.url = row.source_url
        MERGE (c)-[:FILED {date: date(row.filing_date)}]->(f)""",
        rows=MANIFEST,
    )
    session.run(
        """UNWIND $rows AS row
        MATCH (f:Filing {accession_no: row.accession_no})
        MERGE (s:FilingSection {section_key: row.section_key})
        SET s.section_id = row.section_id, s.title = row.title, s.n_chars = row.n_chars
        MERGE (f)-[:HAS_SECTION]->(s)""",
        rows=section_rows,
    )
    n_f = session.run("MATCH (:Filing) RETURN count(*) AS n").single()["n"]
    n_s = session.run("MATCH (:FilingSection) RETURN count(*) AS n").single()["n"]
print(f"{n_f} Filing nodes, {n_s} FilingSection nodes")

5 Filing nodes, 14 FilingSection nodes


## 3. Metric nodes + REPORTS_METRIC edges (XBRL — the no-LLM-numbers rule in action)

`metric_id = {cik}:{metric}:{period_end}`. The `accn` provenance rides on the edge so every number is
traceable to its first-disclosing filing.

In [4]:
metrics = pd.read_parquet(PROJECT_ROOT / "data/processed/xbrl/NVDA_key_metrics.parquet")
metric_rows = [
    {
        "metric_id": f"{int(r.cik)}:{r.metric}:{r.end}",
        "cik": int(r.cik),
        "metric": r.metric,
        "concept": r.concept,
        "value": float(r.val),
        "unit": r.unit,
        "period_start": r.start,
        "period_end": r.end,
        "accn": r.accn,
    }
    for r in metrics.itertuples()
]

with driver.session() as session:
    session.run(
        """UNWIND $rows AS row
        MATCH (c:Company {cik: row.cik})
        MERGE (m:Metric {metric_id: row.metric_id})
        SET m.metric = row.metric, m.concept = row.concept, m.value = row.value, m.unit = row.unit,
            m.period_start = date(row.period_start), m.period_end = date(row.period_end)
        MERGE (c)-[r:REPORTS_METRIC]->(m)
        SET r.accession_no = row.accn""",
        rows=metric_rows,
    )
    n_m = session.run("MATCH (:Metric) RETURN count(*) AS n").single()["n"]
print(f"{n_m} Metric nodes")

60 Metric nodes


In [ ]:
# --- M3 (deterministic layer) assertion cell ---
with driver.session() as session:
    n_companies = session.run("MATCH (c:Company) RETURN count(c) AS n").single()["n"]
    n_filings = session.run("MATCH (:Company {ticker:'NVDA'})-[:FILED]->(f) RETURN count(f) AS n").single()["n"]
    n_sections = session.run("MATCH (:Filing)-[:HAS_SECTION]->(s) RETURN count(DISTINCT s) AS n").single()["n"]
    rev = session.run(
        """MATCH (:Company {ticker:'NVDA'})-[:REPORTS_METRIC]->(m:Metric {metric:'revenue'})
        WHERE m.period_end = date('2024-01-28') RETURN m.value AS v"""
    ).single()
assert n_companies >= 14, f"expected >=14 companies (14 universe + ecosystem later), got {n_companies}"
assert n_filings == 5, f"expected 5 NVDA filings, got {n_filings}"
# one FilingSection node per row of the section-texts store (3 per 10-K, 2 per 10-Q at PoC scope)
assert n_sections == len(section_texts), f"expected {len(section_texts)} sections (from parquet), got {n_sections}"
assert rev and abs(rev["v"] - 60_922_000_000) < 1e6, "FY2024 revenue not queryable from the graph"
driver.close()
print(f"Deterministic layer OK — {n_companies} companies, {n_filings} filings, {n_sections} sections; "
      f"graph answers FY2024 revenue = ${rev['v']/1e9:.3f}B")